# Question 3: When predicting lineup success (NetRtg), how much predictive power is contributed by lineup archetype composition (counts of creators, shooters, rim protectors, defenders, etc.) compared to efficiency metrics (AST%, TS%, REB%, Pace), and which features most strongly drive lineup performance?


In [ ]:

# 1. Imports & paths

import os
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="talk")

# --- paths relative to this notebook ---
LINEUP_DIR = "NBA5ManLineupStats"
ARCH_DIR   = "NBAPlayerArchetypesAssigned"
STATS_DIR  = "NBAPlayerStatsJoined"


In [ ]:

# 2. Helper functions

def season_from_filename(fname: str) -> str:
    """
    Extract '2020-21' from 'NBA5ManLineupStats(2020-21).csv'
    or 'PlayerArchetypes(2020-21).csv', etc.
    """
    if "(" in fname and ")" in fname:
        return fname.split("(")[1].split(")")[0]
    return "Unknown"


def parse_lineup_players(lineup_str: str):
    """
    Split lineup string 'A. Player - B. Player - ...' into a list of tokens.
    """
    if pd.isna(lineup_str):
        return []
    return [p.strip() for p in lineup_str.split(" - ")]


def match_player_full_name(token, team, stats_df, cache, season):
    """
    Map 'S. Curry' + team 'GSW' to a full name in stats_df['Player'].
    Uses a simple heuristic and caches results.
    """
    key = (season, team, token)
    if key in cache:
        return cache[key]

    if pd.isna(token):
        cache[key] = None
        return None

    clean = token.replace(".", "").strip()
    parts = clean.split()
    if not parts:
        cache[key] = None
        return None

    first_init = parts[0][0].lower()
    last_name  = parts[-1].lower()

    # Filter by last name + team if possible
    cand = stats_df[
        stats_df["Player"].str.lower().str.contains(last_name, na=False)
        & (stats_df["Team"] == team)
    ]

    # drop team filter
    if cand.empty:
        cand = stats_df[
            stats_df["Player"].str.lower().str.contains(last_name, na=False)
        ]

    if cand.empty:
        cache[key] = None
        return None

    def first_initial(player_name):
        return player_name.strip()[0].lower() if isinstance(player_name, str) else ""

    cand["first_init_match"] = cand["Player"].apply(first_initial) == first_init

    if cand["first_init_match"].any():
        cand = cand[cand["first_init_match"]]

    if "MP" in cand.columns:
        cand = cand.sort_values("MP", ascending=False)

    full_name = cand.iloc[0]["Player"]
    cache[key] = full_name
    return full_name


def map_archetype_to_role(arch: str) -> str:
    """
    Map your fine-grained archetypes into coarse lineup roles:
        Shooter, Playmaker, Defensive, Scorer, Versatile, Other
    """
    if pd.isna(arch):
        return "Other"

    a = arch.strip().lower()

    # PF/C
    if "stretch big" in a:
        # floor-spacing big, primary value = shooting
        return "Shooter"

    if "rim running shotblocker" in a:
        # lob threat + rim protector
        return "Defensive"

    if "interior creator" in a:
        # post hub, draws doubles, creates offense
        return "Scorer"

    # PG/SG
    if "lead playmaker" in a or "secondary playmaker" in a:
        return "Playmaker"

    if "combo guard" in a:
        # ball-dominant scoring guard
        return "Scorer"

    if "pitbull" in a:
        return "Defensive"

    # SG/SF
    if "3&d specialist" in a:
        #  could argue Shooter or Defensive
        return "Defensive"

    if "2 way player" in a:
        # genuine two-way wing 
        return "Versatile"

    if "slasher" in a:
        return "Scorer"

    if "shooter" in a:
        return "Shooter"

    # SF/PF
    if "point forward" in a or "playmaking forward" in a:
        return "Playmaker"

    # SG/SF/PF
    if "switchable defender" in a:
        return "Defensive"

    if "glue guy" in a:
        return "Versatile"

    if "athletic finisher" in a:
        return "Scorer"

    # PG/SG/SF
    if "microwave" in a:
        return "Scorer"

    # generic fallbacks 
    if "playmaker" in a or "creator" in a or "point forward" in a:
        return "Playmaker"

    if "shooter" in a or "stretch" in a:
        return "Shooter"

    if "defender" in a or "shotblocker" in a or "rim running" in a or "3&d" in a:
        return "Defensive"

    if "slasher" in a or "finisher" in a or "scorer" in a or "microwave" in a:
        return "Scorer"

    if "glue" in a or "versatile" in a or "2 way" in a or "two-way" in a:
        return "Versatile"

    return "Other"


ROLE_GROUPS = ["Shooter", "Playmaker", "Defensive", "Scorer", "Versatile", "Other"]


def adjusted_r2(r2, n_samples, n_features):
    """
    Compute adjusted R².
    """
    if n_samples <= n_features + 1:
        return np.nan
    return 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)



In [ ]:

# 3. Build lineup-level dataset with archetype counts + player stats

all_lineup_rows = []

name_cache = {}

for fname in sorted(os.listdir(LINEUP_DIR)):
    if not fname.endswith(".csv"):
        continue

    season = season_from_filename(fname)
    print(f"Processing season {season}...")

    # load lineup stats for that season 
    lineup_path = os.path.join(LINEUP_DIR, fname)
    ldf = pd.read_csv(lineup_path)

    # cleaning
    ldf["Lineups"] = ldf["Lineups"].astype(str).str.strip()
    ldf["TEAM"]    = ldf["TEAM"].astype(str).str.strip()
    ldf["Poss"] = ldf["PACE"] * (ldf["MIN"] / 48)


    # Drop junk rows
    ldf = ldf[ldf["Lineups"].str.contains(" - ", na=False)].copy()

    # load player stats & archetypes for same season
    stats_fname = f"NBAPlayerStatsJoined({season}).csv"
    stats_path  = os.path.join(STATS_DIR, stats_fname)
    stats_df    = pd.read_csv(stats_path)

    stats_df["Player"] = stats_df["Player"].astype(str).str.strip()
    stats_df["Team"]   = stats_df["Team"].astype(str).str.strip()

    arch_fname = f"PlayerArchetypes({season}).csv"
    arch_path  = os.path.join(ARCH_DIR, arch_fname)
    arch_df    = pd.read_csv(arch_path)
    arch_df["Player"]    = arch_df["Player"].astype(str).str.strip()
    arch_df["Archetype"] = arch_df["Archetype"].astype(str).str.strip()

    # index archetypes by player name for quick lookup
    arch_map = arch_df.set_index("Player")["Archetype"].to_dict()

    for _, row in ldf.iterrows():
        lineup_str = row["Lineups"]
        team       = row["TEAM"]
        players    = parse_lineup_players(lineup_str)

        roles = []

        ts_list      = []
        ast_list     = []
        oreb_list    = []
        dreb_list    = []
        tov_list     = []
        threepar_list = []
        usg_list     = []

        for token in players:
            full_name = match_player_full_name(
                token, team, stats_df, cache=name_cache, season=season
            )

            # Archetype -> role group
            if full_name in arch_map:
                fine_arch = arch_map[full_name]
                role = map_archetype_to_role(fine_arch)
                roles.append(role)
            else:
                roles.append("Other")

            # Individual efficiency stats
            if full_name is not None:
                srow = stats_df[stats_df["Player"] == full_name]
                if not srow.empty:
                    sr = srow.iloc[0]
                    ts_list.append(sr.get("TS%",  np.nan))
                    ast_list.append(sr.get("AST%", np.nan))
                    oreb_list.append(sr.get("ORB%", np.nan))
                    dreb_list.append(sr.get("DRB%", np.nan))
                    tov_list.append(sr.get("TOV%", np.nan))
                    threepar_list.append(sr.get("3PAr", np.nan))
                    usg_list.append(sr.get("USG%", np.nan))

        role_counts = Counter(roles)

        
        def safe_mean(lst):
            return float(np.nanmean(lst)) if len(lst) > 0 else np.nan

        row_dict = {
            "Season": season,
            "TEAM": team,
            "Lineups": lineup_str,
            "MIN": row.get("MIN", np.nan),
            "NetRtg": row.get("NetRtg", np.nan),
            "PACE": row.get("PACE", np.nan),
            "OffRtg": row.get("OffRtg", np.nan),
            "DefRtg": row.get("DefRtg", np.nan),
            "Poss": row.get("Poss", np.nan),


            # lineup-level efficiency 
            "lineup_TS":  row.get("TS%",  np.nan),
            "lineup_AST": row.get("AST%", np.nan),
            "lineup_OREB": row.get("OREB%", np.nan),
            "lineup_DREB": row.get("DREB%", np.nan),

            
            "indiv_AST_mean":   safe_mean(ast_list),
            "indiv_OREB_mean":  safe_mean(oreb_list),
            "indiv_DREB_mean":  safe_mean(dreb_list),
            "indiv_TOV_mean":   safe_mean(tov_list),
            "indiv_3PAr_mean":  safe_mean(threepar_list),
            "indiv_USG_mean":   safe_mean(usg_list),
        }

        # add role counts
        for g in ROLE_GROUPS:
            row_dict[f"count_{g}"] = role_counts.get(g, 0)

        all_lineup_rows.append(row_dict)

lineup_df = pd.DataFrame(all_lineup_rows)
print("Raw lineup_df shape:", lineup_df.shape)

# Drop rows with missing NetRtg
lineup_df = lineup_df.dropna(subset=["NetRtg"]).reset_index(drop=True)


In [ ]:

# 4. Prepare modeling dataset

# filter out extremely low-minute lineups
MIN_MINUTES = 60
lineup_df = lineup_df[lineup_df["MIN"] >= MIN_MINUTES].copy()
print("After MIN filter:", lineup_df.shape)

# predictors
efficiency_cols = [
    "indiv_AST_mean",
    "indiv_OREB_mean",
    "indiv_DREB_mean",
    "indiv_TOV_mean",
    "indiv_3PAr_mean",
    "indiv_USG_mean",
    "PACE",
    "MIN",
]

archetype_cols = [c for c in lineup_df.columns if c.startswith("count_")]

# Drop rows with missing key predictors
model_df = lineup_df.dropna(subset=efficiency_cols).copy()
print("Model_df shape after dropping NA in efficiency stats:", model_df.shape)


features_A = efficiency_cols                    # Model A: efficiency only
features_B = efficiency_cols + archetype_cols   # Model B: efficiency + archetypes

model_df["Poss"] = model_df["PACE"] * (model_df["MIN"] / 48)
STABLE_THRESHOLD = 175   
stable_df = model_df[model_df["Poss"] >= STABLE_THRESHOLD].copy()

X_A = stable_df[features_A].to_numpy()
X_B = stable_df[features_B].to_numpy()

# clip extreme NetRtg
stable_df["NetRtg_clipped"] = stable_df["NetRtg"].clip(-30, 30)
y   = stable_df["NetRtg_clipped"].to_numpy()

possessions = stable_df["Poss"].to_numpy()
possessions = np.clip(possessions, 1e-3, None)

X_train_A, X_test_A, X_train_B, X_test_B, y_train, y_test, w_train, w_test = train_test_split(
    X_A,
    X_B,
    y,
    possessions,
    test_size=0.2,
    random_state=42,
)



print("Train size:", len(y_train), "Test size:", len(y_test))

In [ ]:

# 5. Z-score scaling & outlier clipping

scaler_A = StandardScaler()
scaler_B = StandardScaler()

X_train_A_scaled = scaler_A.fit_transform(X_train_A)
X_test_A_scaled  = scaler_A.transform(X_test_A)

X_train_B_scaled = scaler_B.fit_transform(X_train_B)
X_test_B_scaled  = scaler_B.transform(X_test_B)

# Clip to +/- 3 std to reduce outlier impact
X_train_A_scaled = np.clip(X_train_A_scaled, -3, 3)
X_test_A_scaled  = np.clip(X_test_A_scaled, -3, 3)
X_train_B_scaled = np.clip(X_train_B_scaled, -3, 3)
X_test_B_scaled  = np.clip(X_test_B_scaled, -3, 3)

In [ ]:

# 6. Fit models A & B

rf_A = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
rf_B = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

rf_A.fit(X_train_A_scaled, y_train, sample_weight=w_train)
rf_B.fit(X_train_B_scaled, y_train, sample_weight=w_train)

y_pred_A = rf_A.predict(X_test_A_scaled)
y_pred_B = rf_B.predict(X_test_B_scaled)


In [ ]:

# 7. Metrics: R², adjusted R², MAE
r2_A  = r2_score(y_test, y_pred_A)
mae_A = mean_absolute_error(y_test, y_pred_A, sample_weight=w_test)
adj_A = adjusted_r2(r2_A,
                    n_samples=len(y_test),
                    n_features=X_train_A_scaled.shape[1])

r2_B  = r2_score(y_test, y_pred_B)
mae_B = mean_absolute_error(y_test, y_pred_B, sample_weight=w_test)
adj_B = adjusted_r2(r2_B,
                    n_samples=len(y_test),
                    n_features=X_train_B_scaled.shape[1])

print("=== Model A (Efficiency only) ===")
print("R²:",  r2_A)
print("Adj. R²:", adj_A)
print("MAE:", mae_A)

print("\n=== Model B (Efficiency + Archetypes) ===")
print("R²:",  r2_B)
print("Adj. R²:", adj_B)
print("MAE:", mae_B)

results_table = pd.DataFrame(
    {
        "Model": ["Efficiency Only (A)", "Eff + Archetypes (B)"],
        "R2": [r2_A, r2_B],
        "Adj_R2": [adj_A, adj_B],
        "MAE": [mae_A, mae_B],
    }
)
display(results_table)


In [ ]:

# 8. Permutation importance for Model B (which features matter most)

perm_B = permutation_importance(
    rf_B, X_test_B_scaled, y_test, n_repeats=20, random_state=42, n_jobs=-1
)

imp_df_B = (
    pd.DataFrame(
        {
            "feature": features_B,
            "importance": perm_B.importances_mean,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top permutation importances (Model B):")
display(imp_df_B.head(20))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=imp_df_B.head(10),
    x="importance",
    y="feature",
)
plt.title("Permutation Importance (Model B: Eff + Archetypes)")
plt.xlabel("Mean decrease in R² when shuffled")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:


# 9. Predicted vs Actual NetRtg

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

# Plot A
axes[0].scatter(y_test, y_pred_A, alpha=0.6)
axes[0].axline((0, 0), slope=1, color="red", linestyle="--")
axes[0].set_title(f"Model A: Efficiency Only\n MAE = {mae_A:.2f}")
axes[0].set_xlabel("Actual NetRtg")
axes[0].set_ylabel("Predicted NetRtg")

# Plot B
axes[1].scatter(y_test, y_pred_B, alpha=0.6, color='orange')
axes[1].axline((0, 0), slope=1, color="red", linestyle="--")
axes[1].set_title(f"Model B: Eff + Archetypes\nMAE = {mae_B:.2f}")
axes[1].set_xlabel("Actual NetRtg")

plt.tight_layout()
plt.show()